In [5]:
# all work is done by Colin Houde

# first I imported the necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'torch'

In [ ]:
# the next step was to set up the data transformations and normalization methods

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [ ]:
# loading the data directory
data_dir = '/kaggle/input/human-action-recognition-dataset/Structured'

# loading the images themselve
image_datasets = {
    'train': datasets.ImageFolder(data_dir + '/train', data_transforms['train']),
    'test': datasets.ImageFolder(data_dir + '/test', data_transforms['test']),
}

# loading the dataloaders
batchSizeNum = 128
numWorkersNum = 10
dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=batchSizeNum, shuffle=True, num_workers=numWorkersNum),
    'test': DataLoader(image_datasets['test'], batch_size=batchSizeNum, shuffle=False, num_workers=numWorkersNum)
}


dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'test']}
class_names = image_datasets['train'].classes


In [ ]:
# loading the pretrained resnet 18 model
model = models.resnet18(pretrained=True)


# Unfreeze the last few layers for fine-tuning
for param in model.layer4.parameters():
    param.requires_grad = True

num_ftrs = model.fc.in_features

# adding a linear layer so that there is 15 classes outputs -- as well as batch normalization layer
model.fc = nn.Sequential(
    nn.BatchNorm1d(num_ftrs),
    nn.Linear(num_ftrs, 15)
)


In [ ]:
# setting up the loss and optimizer functions
criterion = nn.CrossEntropyLoss()

# using a SDG optimizer
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [ ]:
# setting up the training loop

import time
import matplotlib.pyplot as plt
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()  # For mixed precision

def train_model(model, criterion, optimizer, num_epochs=32, log_filename='training_log.txt'):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    since = time.time()

    # Initialize lists to store losses and times for plotting
    train_losses = []
    test_losses = []
    epoch_times = []

    # Open the log file to store training progress
    with open(log_filename, 'w') as log_file:
        log_file.write('Training Log\n')
        log_file.write('=' * 50 + '\n')

        for epoch in range(num_epochs):
            epoch_start = time.time()

            log_file.write(f'Epoch {epoch}/{num_epochs - 1}\n')
            log_file.write('-' * 10 + '\n')
            
            epoch_train_loss = 0.0
            epoch_test_loss = 0.0

            for phase in ['train', 'test']:
                if phase == 'train':
                    model.train()  # Set model to training mode
                else:
                    model.eval()  # Set model to evaluate mode

                running_loss = 0.0
                running_corrects = 0

                for inputs, labels in dataloaders[phase]:
                    inputs, labels = inputs.to(device), labels.to(device)

                    optimizer.zero_grad()

                    with torch.set_grad_enabled(phase == 'train'):
                        with autocast():  # Mixed precision training
                            outputs = model(inputs)
                            loss = criterion(outputs, labels)
                            _, preds = torch.max(outputs, 1)

                        if phase == 'train':
                            scaler.scale(loss).backward()
                            scaler.step(optimizer)
                            scaler.update()

                    running_loss += loss.item() * inputs.size(0)
                    running_corrects += torch.sum(preds == labels.data)

                epoch_loss = running_loss / dataset_sizes[phase]
                epoch_acc = running_corrects.double() / dataset_sizes[phase]

                # Log the loss and accuracy
                log_file.write(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}\n')

                # Store the loss for plotting
                if phase == 'train':
                    train_losses.append(epoch_loss)
                    epoch_train_loss = epoch_loss
                else:
                    test_losses.append(epoch_loss)
                    epoch_test_loss = epoch_loss

            # Record time for this epoch
            epoch_time = time.time() - epoch_start
            epoch_times.append(epoch_time)
            log_file.write(f'Time for epoch {epoch}: {epoch_time // 60:.0f}m {epoch_time % 60:.0f}s\n\n')

        # Calculate and log total training time
        total_time = time.time() - since
        log_file.write(f'Training complete in {total_time // 60:.0f}m {total_time % 60:.0f}s\n')

    # Plot the results after training is complete
    plot_training_results(train_losses, test_losses, epoch_times)

    return model


def plot_training_results(train_losses, test_losses, epoch_times):
    epochs = range(1, len(train_losses) + 1)

    # Plot loss curves
    plt.figure(figsize=(14, 5))

    # Loss graph
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, 'b-', label='Train Loss')
    plt.plot(epochs, test_losses, 'r-', label='Test Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train vs Test Loss')
    plt.legend()

    # Time per epoch graph -- comment out if you want the time per epoch but total time taken is a better metric which is already outputted
#     plt.subplot(1, 2, 2)
#     plt.plot(epochs, epoch_times, 'g-', label='Epoch Time (seconds)')
#     plt.xlabel('Epoch')
#     plt.ylabel('Time (seconds)')
#     plt.title('Time Taken per Epoch')
#     plt.legend()

    plt.tight_layout()
    plt.show()

trained_model = train_model(model, criterion, optimizer, num_epochs=32)

In [ ]:
# setting up the testing loop
import random
import matplotlib.pyplot as plt
import numpy as np

def evaluate_model(model, dataloader, dataset_size, num_images=10):
    model.eval()  # Set the model to evaluation mode
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    
    correct_preds = 0
    total_preds = 0
    images_to_show = []

    fig = plt.figure(figsize=(20, 10))  # Create a figure for displaying the images
    model.to(device)

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            # Count correct predictions for accuracy
            correct_preds += torch.sum(preds == labels).item()
            total_preds += labels.size(0)

            # Store the images, predictions, and true labels for later random display
            for i in range(inputs.size(0)):
                images_to_show.append((inputs[i].cpu(), preds[i].cpu(), labels[i].cpu()))

    # Calculate and display the accuracy
    accuracy = correct_preds / total_preds * 100
    print(f'Accuracy on the test set: {accuracy:.2f}%')

    # Randomly select 10 images from the list
    selected_images = random.sample(images_to_show, num_images)

    # Display the selected images and their predicted and true labels
    images_so_far = 0
    for idx, (img, pred, label) in enumerate(selected_images):
        images_so_far += 1
        ax = plt.subplot(num_images // 2, 5, images_so_far)
        ax.axis('off')
        ax.set_title(f'Predicted: {pred.item()}, True: {label.item()}')

        # Convert the image tensor to numpy format for plotting
        img_np = img.permute(1, 2, 0).numpy()  # Convert CHW to HWC format
        img_np = np.clip(img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]), 0, 1)  # Unnormalize
        ax.imshow(img_np)

        if images_so_far == num_images:
            break

    plt.show()
    
evaluate_model(model, dataloaders['test'], dataset_sizes['test'], num_images=10)
